In [0]:

%python
# Databricks notebook in Python / PySpark

# -------------------------------
# Pas 1: Import functii uzuale
# -------------------------------
from pyspark.sql.functions import col, when, concat_ws, lit, sum, split

# -------------------------------
# Pas 2: Incarcare tabele Sample
# -------------------------------
customers = spark.table("samples.tpch.customer")
orders = spark.table("samples.tpch.orders")
products = spark.table("samples.tpch.part")

# -------------------------------
# Pas 3: Vizualizare schema si primele randuri
# -------------------------------
print("Customers Schema")
customers.printSchema()
customers.show(5)

print("Orders Schema")
orders.printSchema()
orders.show(5)

print("Products Schema")
products.printSchema()
products.show(5)


Customers Schema
root
 |-- c_custkey: long (nullable = true)
 |-- c_name: string (nullable = true)
 |-- c_address: string (nullable = true)
 |-- c_nationkey: long (nullable = true)
 |-- c_phone: string (nullable = true)
 |-- c_acctbal: decimal(18,2) (nullable = true)
 |-- c_mktsegment: string (nullable = true)
 |-- c_comment: string (nullable = true)

+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|c_custkey|            c_name|           c_address|c_nationkey|        c_phone|c_acctbal|c_mktsegment|           c_comment|
+---------+------------------+--------------------+-----------+---------------+---------+------------+--------------------+
|   412445|Customer#000412445|0QAB3OjYnbP6mA0B,kgf|         21|31-421-403-4333|  5358.33|    BUILDING|arefully blithely...|
|   412446|Customer#000412446|5u8MSbyiC7J,7PuY4...|         20|30-487-949-7942|  9441.59|   MACHINERY|sleep according t...|
|   412447|Customer#000412

In [0]:
%python

# -------------------------------
# Pas 4: Transformari pe customers
# -------------------------------
customers_transformed = customers.select(
    col("c_custkey").alias("customer_id"),
    split(col("c_name"), " ").getItem(0).alias("first_name"),
    split(col("c_name"), " ").getItem(1).alias("last_name"),
    col("c_name").alias("full_name")
).withColumn(
    "CustomerType",
    when(col("customer_id") < 1000, "Regular").otherwise("VIP")
)

customers_transformed.show(5)

+-----------+------------------+---------+------------------+------------+
|customer_id|        first_name|last_name|         full_name|CustomerType|
+-----------+------------------+---------+------------------+------------+
|     412445|Customer#000412445|     NULL|Customer#000412445|         VIP|
|     412446|Customer#000412446|     NULL|Customer#000412446|         VIP|
|     412447|Customer#000412447|     NULL|Customer#000412447|         VIP|
|     412448|Customer#000412448|     NULL|Customer#000412448|         VIP|
|     412449|Customer#000412449|     NULL|Customer#000412449|         VIP|
+-----------+------------------+---------+------------------+------------+
only showing top 5 rows


In [0]:
%python
# -------------------------------
# Pas 5: Filtrare si sortare comenzi
# -------------------------------
orders_filtered = orders.filter(col("o_totalprice") > 500).orderBy(col("o_totalprice").desc())
orders_filtered.show(5)

# -------------------------------
# Pas 6: Join clienti + comenzi
# -------------------------------
customer_orders = customers_transformed.join(
    orders_filtered,
    customers_transformed.customer_id == orders_filtered.o_custkey,
    "inner"
).select(
    customers_transformed.full_name,
    "CustomerType",
    col("o_orderkey").alias("order_id"),
    col("o_totalprice").alias("total_amount")
)

customer_orders.show(5)

+----------+---------+-------------+------------+-----------+---------------+---------------+--------------+--------------------+
|o_orderkey|o_custkey|o_orderstatus|o_totalprice|o_orderdate|o_orderpriority|        o_clerk|o_shippriority|           o_comment|
+----------+---------+-------------+------------+-----------+---------------+---------------+--------------+--------------------+
|   2199712|   333941|            O|   569370.40| 1996-09-30|         2-HIGH|Clerk#000003248|             0| the final, ironi...|
|  18869634|   565334|            F|   565520.21| 1995-01-10|       3-MEDIUM|Clerk#000000447|             0|carefully. always...|
|   5200102|   466960|            O|   550628.34| 1997-01-22|         2-HIGH|Clerk#000000458|             0|n instructions. u...|
|   2745894|   322406|            O|   549127.10| 1996-07-04|4-NOT SPECIFIED|Clerk#000003309|             0|l attainments. fu...|
|  16089345|   185383|            O|   539520.20| 1996-07-09|         2-HIGH|Clerk#0000016

In [0]:
%python

# -------------------------------
# Pas 7: Agregari / GroupBy
# -------------------------------
customer_orders.groupBy("CustomerType").agg(
    sum("total_amount").alias("TotalSpent")
).show()

# -------------------------------
# Pas 8: Creare coloana cu conditii
# -------------------------------
customer_orders = customer_orders.withColumn(
    "OrderCategory",
    when(col("total_amount") > 1000, "High Value")
    .when(col("total_amount") > 500, "Medium Value")
    .otherwise("Low Value")
)

customer_orders.show(5)


+------------+----------------+
|CustomerType|      TotalSpent|
+------------+----------------+
|     Regular|   1492081415.40|
|         VIP|1131947133830.85|
+------------+----------------+

+------------------+------------+--------+------------+-------------+
|         full_name|CustomerType|order_id|total_amount|OrderCategory|
+------------------+------------+--------+------------+-------------+
|Customer#000000019|     Regular|10781955|   245464.69|   High Value|
|Customer#000000019|     Regular|21733377|   230620.50|   High Value|
|Customer#000000019|     Regular|22059172|   159415.82|   High Value|
|Customer#000000019|     Regular| 3951331|    74035.02|   High Value|
|Customer#000000019|     Regular|  164711|   331023.19|   High Value|
+------------------+------------+--------+------------+-------------+
only showing top 5 rows


In [0]:
%python
# -------------------------------
# Pas 9: Distinct, drop, rename
# -------------------------------
customers_transformed.select("customer_id", "full_name").distinct().show(5)
customers_transformed.withColumnRenamed("full_name", "CustomerFullName").show(5)
customers_transformed.drop("first_name").show(5)

+-----------+------------------+
|customer_id|         full_name|
+-----------+------------------+
|     412916|Customer#000412916|
|     413029|Customer#000413029|
|     413113|Customer#000413113|
|     413415|Customer#000413415|
|     413423|Customer#000413423|
+-----------+------------------+
only showing top 5 rows
+-----------+------------------+---------+------------------+------------+
|customer_id|        first_name|last_name|  CustomerFullName|CustomerType|
+-----------+------------------+---------+------------------+------------+
|     412445|Customer#000412445|     NULL|Customer#000412445|         VIP|
|     412446|Customer#000412446|     NULL|Customer#000412446|         VIP|
|     412447|Customer#000412447|     NULL|Customer#000412447|         VIP|
|     412448|Customer#000412448|     NULL|Customer#000412448|         VIP|
|     412449|Customer#000412449|     NULL|Customer#000412449|         VIP|
+-----------+------------------+---------+------------------+------------+
only